In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from index_evaluator import load_authors_data, evaluate_all_with_expert_index
from data_utils import load_author_publications
from index_metrics import (
    h_index, g_index, i10_index, ha_index, ar_index,
    timed_h_index, contemporary_h_index, trend_h_index,
    career_years_h_index_by_average_citations_per_year,
)
from viz_utils import *

In [ ]:
# 기본 설정
authors_file = '../gsc_data/authors.all'
data_dir = '../gsc_data/DATA/'
start_year = 2013
end_year = 2013

# 전체 저자 메타 데이터 불러오기
authors_data = load_authors_data(authors_file)

# 평가할 분야 리스트
fields = ["computer_science", "biomedical", "humanities", "physical"]

# 루프 실행
for field in fields:
    print(f"\n📁 Setting up field: {field.title().replace('_', ' ')}")

    # ⬇️ 분야별 디렉토리 생성
    output_dir = f"results_by_field/{field}"
    os.makedirs(output_dir, exist_ok=True)

    # 분야별 저자 ID 로드
    author_file = f"authors_{field}.csv"
    author_ids = pd.read_csv(author_file)["author_id"].tolist()

    # 해당 분야 저자만 필터링
    field_data = authors_data[authors_data["author_id"].isin(author_ids)]

    for year in range(start_year, end_year + 1):
        print(f"🧪 {field} | Year: {year}")
        output_file = f"{output_dir}/index_by_{year}_{field}.csv"

        # 이미 있으면 스킵
        if os.path.exists(output_file):
            print(f"⏭️  Already exists: {output_file}")
            continue

        # 지수 계산
        index_df, max_q, max_r = evaluate_all_with_expert_index(field_data, data_dir, current_year=year)

        # 저장
        index_df.to_csv(output_file, index=False)
        print(f"✅ Saved: {output_file}")


In [ ]:
import pandas as pd

def load_index_df_by_field_and_year(field: str, year: int, root_dir: str = "./results_by_field/") -> pd.DataFrame:
    """
    주어진 분야와 연도에 해당하는 결과 파일(index_by_<year>_<field>.csv)을 불러와서 DataFrame으로 반환.
    
    Parameters:
        field (str): 분야 이름 (예: 'computer_science')
        year (int): 연도 (예: 2013)
        root_dir (str): 저장된 결과의 루트 폴더
    
    Returns:
        pd.DataFrame: 해당 분야/연도의 지수 결과 DataFrame
    """
    file_path = f"{root_dir}{field}/index_by_{year}_{field}.csv"
    try:
        df = pd.read_csv(file_path)
        print(f"✅ Loaded: {file_path}")
        return df
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return pd.DataFrame()


In [ ]:
# fields = ["computer_science", "biomedical", "humanities", "physical"]

index_df = load_index_df_by_field_and_year("physical", 2013)


# 2. 컬럼 이름 축약 및 분석 대상 설정
index_df_short = index_df.rename(columns=INDEX_NAME_MAP)
index_columns = list(INDEX_NAME_MAP.values())

In [ ]:
# 3. 분석 및 시각화 실행
print("📊 상관관계 분석 중...")
plot_index_scatter(index_df, "expert_index", "h_index")
plot_index_scatter(index_df, "expert_index", "g_index")
plot_index_scatter(index_df, "expert_index", "i10_index")
plot_index_scatter(index_df, "expert_index", "contemporary_h_index")
plot_index_scatter(index_df, "expert_index", "ar_index")
plot_index_scatter(index_df, "expert_index", "hpd_index")
plot_index_scatter(index_df, "expert_index", "career_year_h_index_by_publications")
plot_index_scatter(index_df, "expert_index", "career_year_h_index_by_publications_year_citations")
plot_index_scatter(index_df, "expert_index", "career_years_h_index_by_average_citations_per_year")
plot_index_scatter(index_df, "expert_index", "timed_h_index_5")
plot_index_scatter(index_df, "expert_index", "timed_h_index_10")
plot_index_scatter(index_df, "expert_index", "ha_index")


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

corr_pearson = plot_index_correlation_heatmap(index_df, method='pearson')
print(corr_pearson)
mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)

# 히트맵 출력
plt.figure(figsize=(12, 10))
sns.heatmap(corr_pearson, annot=True, annot_kws={"size": 10}, cmap="coolwarm", mask=mask)
plt.title("Pearson Correlation between Indices", fontsize=14)
plt.tight_layout()
plt.savefig("pearson_heatmap.png", dpi=600)

corr_spearman = plot_index_correlation_heatmap(index_df, method='spearman')
print(corr_spearman)

mask = np.triu(np.ones_like(corr_spearman, dtype=bool), k=1)
plt.figure(figsize=(12, 10))
sns.heatmap(corr_spearman, annot=True, annot_kws={"size": 10}, cmap="coolwarm", mask=mask)
plt.title("Pearson Correlation between Indices", fontsize=14)
plt.tight_layout()
plt.savefig("spearman_heatmap.png", dpi=600)


In [ ]:
print("🔝 Top-K 비교 및 겹침 분석 중...")
# print_top_k_authors(index_df_short, k=1000, index_columns=index_columns)
overlap_matrix = compute_top_k_overlap_matrix(index_df_short, index_columns, k=800)
print(overlap_matrix)
mask = np.triu(np.ones_like(overlap_matrix, dtype=float), k=1)
plt.figure(figsize=(12, 10))
sns.heatmap(overlap_matrix.astype(float), annot=True, fmt=".2f", annot_kws={"size": 10}, cmap="YlGnBu", mask=mask)
plt.title("Top-K Author Overlap Between Indices", fontsize=14)
plt.tight_layout()
# plt.savefig("top_k_overlap_heatmap.png", dpi=600)

In [ ]:
print("🧍 독립 저자 분석 중...")
unique_authors_by_index = find_index_specific_top_k_authors(index_df_short, index_columns, k=20)
for idx, authors in unique_authors_by_index.items():
    print(f"🔹 {idx} 지수에만 Top-K로 포함된 저자 {len(authors)}명: {authors}")


In [ ]:
print("🔢 Top-K 확장에 따른 고유 저자 수 시각화 중...")
k_range = list(range(10, 110, 10))
author_counts_by_k = plot_unique_author_counts_by_k(index_df_short, index_columns, k_values=k_range)
# print(author_counts_by_k)

In [ ]:
from data_utils import load_author_index_time_series
from viz_utils import plot_author_index_and_activity_time_series

author_id = "Ll-b-IwAAAAJ"
index_df_single = load_author_index_time_series(author_id, "./", 1970, 2013)
plot_df, paper_stats = plot_author_index_and_activity_time_series(author_id, index_df_single, data_dir)


In [ ]:
print(paper_stats)

In [ ]:
from collections import defaultdict
import numpy as np

# --- 2. 설정 ---
current_year = 2013

summary_stats = []
temporal_activity = defaultdict(list)

top_k_dict = get_top_k_authors(index_df, k=20, index_cols=None)

# --- 3. 데이터 분석 ---
for index_name, author_ids in top_k_dict.items():
    for author_id in author_ids:
        df = load_author_publications(data_dir, author_id, current_year)
        df = df[(df["year"] >= 1970) & (df["year"] <= current_year)]  # ✅ 연도 필터링
        if df.empty:
            continue

        # 기초 통계
        summary_stats.append({
            "index": index_name,
            "author_id": author_id,
            "papers": len(df),
            "total_citations": df["citations"].sum(),
            "avg_citations": df["citations"].mean(),
            "recent_5y_citations": df[df["year"] >= current_year - 5]["citations"].sum(),
            "active_years": df["year"].nunique(),
            "avg_year": df["year"].mean()
        })

        # 연도별 인용 수 저장
        year_group = df.groupby("year")["citations"].sum()
        for year, c in year_group.items():
            temporal_activity[(index_name, year)].append(c)

# --- 4. 통계 테이블 생성 ---
summary_df = pd.DataFrame(summary_stats)
print("\n📊 [Summary of Basic Statistics for Independent Authors]")
print(summary_df.groupby("index").agg({
    "papers": ["mean", "std"],
    "total_citations": ["mean", "std"],
    "avg_citations": ["mean", "std"],
    "recent_5y_citations": ["mean", "std"],
    "active_years": ["mean", "std"],
    "avg_year": ["mean", "std"]
}).round(2))

# --- 5. 시간 추세 분석용 데이터 구성 ---
temporal_summary = defaultdict(list)
for (index_name, year), cits in temporal_activity.items():
    temporal_summary["index"].append(index_name)
    temporal_summary["year"].append(int(year))
    temporal_summary["avg_citations"].append(np.mean(cits))

temporal_df = pd.DataFrame(temporal_summary)

# --- 6. 시각화: 시간에 따른 평균 인용 수 변화 ---
# exclude할 컬럼 정의
ordered_indices = ["h", "g", "i10", "hc", "ar", "hpd", "cy_pub", "cy_cit", "cy_avg", "h_t5", "h_t10", "ha", "expert"]
exclude_indices = {"quality_score", "recency_score"}

# 고유한 지수 리스트에서 제외 후 정렬
index_columns = set(temporal_df["index"].unique()) - exclude_indices

style_map = get_styled_index_map(index_columns)

plt.figure(figsize=(12, 6))

# 각 지수별로 스타일 입혀서 시각화
for index_name in ordered_indices:
    if index_name in exclude_indices:
        continue
    group = temporal_df[temporal_df["index"] == index_name].sort_values("year")
    if group.empty:
        continue
    
    style = style_map.get(index_name, {})
    plt.plot(
        group["year"],
        group["avg_citations"],
        label=index_name,
        color=style.get("color", None),
        linestyle=style.get("linestyle", "-"),
        marker=style.get("marker", "o")
    )

plt.title("Average Yearly Citations of Independent Authors")
plt.ylabel("Average Citations per Year")
plt.xlabel("Year")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print(temporal_df)

In [ ]:
for index_name in ordered_indices:
    if index_name in exclude_indices:
        continue
    group = temporal_df[temporal_df["index"] == index_name].sort_values("year")
    if group.empty:
        continue

    style = style_map.get(index_name, {})
    plt.figure(figsize=(10, 4))
    plt.plot(
        group["year"],
        group["avg_citations"],
        label=index_name,
        color=style.get("color", None),
        linestyle=style.get("linestyle", "-"),
        marker=style.get("marker", "o")
    )
    plt.title(f"Yearly Average Citations for Index: {index_name}")
    plt.xlabel("Year")
    plt.ylabel("Average Citations")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()



In [ ]:
import pandas as pd
from data_utils import load_author_publications

def show_author_summary(author_id, index_df, data_dir, current_year=2013):
    """
    주어진 author_id에 대해 각 지수의 점수, 랭킹, 논문 수, 인용 수, 연구 경력을 표로 반환

    Parameters:
    - author_id: str, 저자 ID
    - index_df: DataFrame, 지수 점수 및 author_id를 포함한 데이터프레임
    - data_dir: str, 논문 데이터 디렉토리
    - current_year: int, 기준 연도

    Returns:
    - summary_df: pd.DataFrame, 결과 테이블
    """

    if author_id not in index_df["author_id"].values:
        print(f"❌ 해당 author_id({author_id})는 index_df에 존재하지 않습니다.")
        return

    # 1. 지수 점수 및 랭킹 계산
    row = index_df[index_df["author_id"] == author_id].squeeze()
    rank_df = index_df.copy()
    for col in index_df.columns:
        if col not in {"author_id", "quality_score", "recency_score"} and pd.api.types.is_numeric_dtype(index_df[col]):
            rank_df[f"{col}_rank"] = index_df[col].rank(method="min", ascending=False)
    row_rank = rank_df[rank_df["author_id"] == author_id].squeeze()

    # 2. 논문 데이터 로드
    df = load_author_publications(data_dir, author_id, current_year=current_year)
    if df.empty or "year" not in df.columns:
        print("⚠️ 논문 데이터가 존재하지 않습니다.")
        return

    first_year = int(df["year"].min())
    last_year = int(df["year"].max())
    career_years = last_year - first_year + 1
    paper_count = len(df)
    total_citations = int(df["citations"].sum())

    # 3. 결과 정리
    metrics = [
        col for col in index_df.columns 
        if col not in {"author_id", "quality_score", "recency_score"}
        and pd.api.types.is_numeric_dtype(index_df[col])
    ]

    data = {
        "author_id": author_id,
        "paper_count": paper_count,
        "citation_count": total_citations,
        "first_pub_year": first_year,
        "last_pub_year": last_year,
        "career_years": career_years
    }

    for col in metrics:
        data[col] = row[col]
        data[f"{col}_rank"] = int(row_rank[f"{col}_rank"])

    return pd.DataFrame([data])


In [ ]:
# unique_authors_by_index 딕셔너리 → 중복 없이 author_id 리스트
all_unique_authors = sorted(set().union(*unique_authors_by_index.values()))

summary_dfs = []

for author_id in all_unique_authors:
    df = show_author_summary(author_id, index_df, data_dir)
    if df is not None:
        summary_dfs.append(df)

# 전체 결과 통합
final_summary = pd.concat(summary_dfs, ignore_index=True)

# 결과 미리보기
display(final_summary.head())

output_path = "unique_authors_summary.csv"

# 저장
final_summary.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ 저장 완료: {output_path}")
